# Unified Data Loader

This notebook compares echograms loaded from the same `.raw` file in two ways:
uncalibrated raw power via `load_raw()`, and calibrated Sv via `load_zarr()` after
converting with `raw_to_zarr()` (echopype). Both return the same `EchogramData` container,
so the downstream detection code is identical.

The CRIMAC preprocessor stores Sv as **linear** values (not dB). The `sv` variable in the
zarr contains values like `4e-08` which correspond to about `-74 dB re 1 m⁻¹`. This is
standard — echopype stores Sv directly in dB, while CRIMAC stores it in linear and our
loader converts both to dB.

In [ ]:
import sys
sys.path.insert(0, '../..')
from project_paths import REPO_ROOT

import numpy as np
import matplotlib.pyplot as plt

from raw_sensor_integration.data_loader import (
    load_raw, load_zarr, convert_raw_dir,
)
from raw_sensor_integration import bottom_detection

In [ ]:
RAW_DIR = REPO_ROOT / 'Datasets/raw_files/drive-download-20260317T113707Z-1-001'
ZARR_DIR = RAW_DIR / 'zarr'
FREQUENCY = 38000

raw_files = sorted(RAW_DIR.glob('*.raw'))
zarr_files = sorted(ZARR_DIR.glob('*.zarr'))
print(f'{len(raw_files)} raw files, {len(zarr_files)} zarr files')

## Convert raw files to zarr (skip existing)

If the zarr files already exist this cell finishes instantly.

In [ ]:
zarr_paths = convert_raw_dir(RAW_DIR, output_dir=ZARR_DIR, skip_existing=True)

## Side-by-side comparison: raw vs calibrated zarr

Loading the same `.raw` file both ways lets us see the effect of calibration.
The uncalibrated raw power is clipped to `[-100, 0] dB` and uses an adaptive
per-ping threshold, while the calibrated Sv from echopype has physically
meaningful dB values and uses the standard `-31 dB` threshold.

In [ ]:
def detect_bottom(data):
    """Run simple + edge detectors and ensemble them."""
    kwargs = dict(
        power_db=data.sv_db,
        range_array=data.range_array,
        heave=data.heave,
        pulse_duration=data.pulse_duration,
        threshold_log_sv=data.default_threshold,
    )
    b_simple = bottom_detection.detect_bottom_simple(**kwargs)
    b_edge = bottom_detection.detect_bottom_edge(**kwargs)
    return bottom_detection.ensemble_bottom([b_simple, b_edge])

In [ ]:
# Pick a file to compare (change FILE_IDX to try different files)
FILE_IDX = 3

raw_path = raw_files[FILE_IDX]
zarr_path = ZARR_DIR / raw_path.with_suffix('.zarr').name

raw_data = load_raw(raw_path, frequency=FREQUENCY)
zarr_data = load_zarr(zarr_path, frequency=FREQUENCY)

print(f'File: {raw_path.name}')
print(f'Raw:  {raw_data.sv_db.shape}  Sv=[{np.nanmin(raw_data.sv_db):.0f}, {np.nanmax(raw_data.sv_db):.0f}] dB')
print(f'Zarr: {zarr_data.sv_db.shape}  Sv=[{np.nanmin(zarr_data.sv_db):.0f}, {np.nanmax(zarr_data.sv_db):.0f}] dB')

In [ ]:
raw_bottom = detect_bottom(raw_data)
zarr_bottom = detect_bottom(zarr_data)

for label, data, bottom in [('Raw', raw_data, raw_bottom), ('Zarr', zarr_data, zarr_bottom)]:
    valid = np.sum(~np.isnan(bottom))
    depths = bottom[~np.isnan(bottom)]
    print(f'{label:5s}: {valid}/{data.n_pings} valid ({100*valid/data.n_pings:.1f}%)  '
          f'median depth: {np.median(depths):.1f} m')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

for ax, (data, bottom, label) in zip(axes, [
    (raw_data, raw_bottom, 'Raw (uncalibrated)'),
    (zarr_data, zarr_bottom, 'echopype zarr (calibrated Sv)'),
]):
    vmin, vmax = data.default_vmin_vmax
    im = ax.imshow(data.sv_db, aspect='auto', cmap='viridis',
                   vmin=vmin, vmax=vmax,
                   extent=[0, data.n_pings,
                           data.range_array[-1], data.range_array[0]])
    valid = ~np.isnan(bottom)
    ax.plot(np.where(valid)[0], bottom[valid], 'r-', linewidth=0.8, alpha=0.9)
    unit = 'dB re 1 m\u207b\u00b9' if data.calibrated else 'dB (uncalibrated)'
    ax.set_title(f'{label}  \u2014  {data.frequency_hz/1000:.0f} kHz')
    ax.set_ylabel('Range (m)')
    plt.colorbar(im, ax=ax, label=unit)

axes[1].set_xlabel('Ping')
plt.suptitle(raw_path.name, fontsize=12)
plt.tight_layout()
plt.show()

## Bottom depth comparison across the file

Plotting the raw and zarr bottom depths on the same axis shows how close they are.
Small differences come from calibration shifting the Sv profile, which moves the
detected leading edge by a few meters.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))

pings = np.arange(raw_data.n_pings)
ax.plot(pings, raw_bottom, 'b-', linewidth=0.6, alpha=0.7, label='Raw (auto threshold)')
ax.plot(pings, zarr_bottom, 'r-', linewidth=0.6, alpha=0.7, label='Zarr (-31 dB threshold)')

ax.set_xlabel('Ping')
ax.set_ylabel('Bottom depth (m)')
ax.invert_yaxis()
ax.legend()
ax.set_title(f'Bottom detection comparison — {raw_path.name}')
plt.tight_layout()
plt.show()

# Difference statistics
both_valid = ~np.isnan(raw_bottom) & ~np.isnan(zarr_bottom)
diff = zarr_bottom[both_valid] - raw_bottom[both_valid]
print(f'Difference (zarr - raw): mean={diff.mean():.2f} m, '
      f'std={diff.std():.2f} m, median={np.median(diff):.2f} m')

## All files overview

Quick check across all files in the folder to verify the conversion is consistent.

In [ ]:
print(f'{"File":>25s}  {"Raw pings":>10s}  {"Raw det%":>8s}  {"Zarr det%":>9s}  {"Med diff (m)":>12s}')
print('-' * 75)

for raw_f in raw_files:
    zarr_f = ZARR_DIR / raw_f.with_suffix('.zarr').name
    if not zarr_f.exists():
        continue

    rd = load_raw(raw_f, frequency=FREQUENCY, include_gps=False)
    zd = load_zarr(zarr_f, frequency=FREQUENCY)

    rb = detect_bottom(rd)
    zb = detect_bottom(zd)

    rv = 100 * np.sum(~np.isnan(rb)) / rd.n_pings
    zv = 100 * np.sum(~np.isnan(zb)) / zd.n_pings

    both = ~np.isnan(rb) & ~np.isnan(zb)
    med_diff = np.median(zb[both] - rb[both]) if both.sum() > 0 else float('nan')

    print(f'{raw_f.name:>25s}  {rd.n_pings:>10d}  {rv:>7.1f}%  {zv:>8.1f}%  {med_diff:>11.2f} m')